In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from nn_utils import split_and_shuffle, evaluate_conv_scalar,train_unet
import os
from global_vars import *


train_folder = os.path.join(data_folder, 'nodeep_train_data_ver1/')
test_folder = os.path.join(data_folder, 'nodeep_test_data_ver1/')


X_train_val = pd.read_parquet(os.path.join(train_folder, "X_train.parquet"),engine="fastparquet")
y_t_train_val = pd.read_parquet(os.path.join(train_folder, "y_t_train.parquet"),engine="fastparquet")
y_q_train_val = pd.read_parquet(os.path.join(train_folder, "y_q_train.parquet"),engine="fastparquet")

X_test = pd.read_parquet(os.path.join(test_folder, "X_test.parquet"),engine="fastparquet")
y_t_test = pd.read_parquet(os.path.join(test_folder, "y_t_test.parquet"),engine="fastparquet")
y_q_test = pd.read_parquet(os.path.join(test_folder, "y_q_test.parquet"),engine="fastparquet")

X_train, y_train, X_val, y_val = split_and_shuffle(
    X_train_val.values, y_t_train_val.values, val_frac=0.2, seed=0
)


n_surf_features = len(surf_input_var_names)
n_total_features = X_train.shape[1]
branch2_start = (n_total_features-n_surf_features)

Xc_train_flat = X_train[:, range(branch2_start)]
Xs_train = X_train[:, range(branch2_start, n_total_features)]

Xc_val_flat = X_val[:, range(branch2_start)]
Xs_val = X_val[:, range(branch2_start, n_total_features)]

Xc_test_flat = X_test.values[:, range(branch2_start)]
Xs_test = X_test.values[:, range(branch2_start, n_total_features)]

# Unflatten Xc
n_samples_tr = Xc_train_flat.shape[0]
n_samples_va = Xc_val_flat.shape[0]
n_samples_te = Xc_test_flat.shape[0]
n_conv_vars = 4
n1 = 58
Xc_train = Xc_train_flat.reshape(n_samples_tr, n_conv_vars, n1)
Xc_val = Xc_val_flat.reshape(n_samples_va, n_conv_vars, n1)
Xc_test = Xc_test_flat.reshape(n_samples_te, n_conv_vars, n1)

In [ ]:

model, scaler, history = train_unet(
    Xc_train, Xs_train, y_train,
    Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
    model_args={
        "base_channels": 32,
        "depth":         2, # try depth=2 # depth=3 gave 3.4e-01 after 199eps
        "kernel_size":   3,
        "batchnorm":     True,
        "dropout":       0.1,
        "scalar_embed":  16,
    },
    epochs=200, patience=15, batch_size=256,
)

metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

epoch   0 | train 6.6044e-01 | val 4.7015e-01
epoch   1 | train 5.8183e-01 | val 4.5988e-01
epoch   2 | train 5.5751e-01 | val 4.3672e-01
epoch   3 | train 5.4528e-01 | val 4.0331e-01
epoch   4 | train 5.3345e-01 | val 4.3068e-01
epoch   5 | train 5.2577e-01 | val 4.4870e-01
epoch   6 | train 5.2126e-01 | val 3.9276e-01
epoch   7 | train 5.1472e-01 | val 3.9239e-01
epoch   8 | train 5.1109e-01 | val 3.8075e-01
epoch   9 | train 5.0706e-01 | val 3.9316e-01
epoch  10 | train 5.0599e-01 | val 3.7680e-01
epoch  11 | train 4.9984e-01 | val 3.8569e-01
epoch  12 | train 4.9902e-01 | val 3.7237e-01
epoch  13 | train 4.9545e-01 | val 3.7394e-01
epoch  14 | train 4.9535e-01 | val 3.8286e-01
epoch  15 | train 4.9087e-01 | val 3.6864e-01
epoch  16 | train 4.9037e-01 | val 3.6970e-01
epoch  17 | train 4.8761e-01 | val 3.7274e-01
epoch  18 | train 4.8495e-01 | val 3.6285e-01
epoch  19 | train 4.8502e-01 | val 3.6956e-01
epoch  20 | train 4.8199e-01 | val 3.6843e-01
epoch  21 | train 4.8163e-01 | val

In [ ]:
lst1 = [16,32,64,128]
lst2 = [8,16,32,64]

for a in lst1:
    for b in lst2:
        print(a,b)

        model, scaler, history = train_unet(
            Xc_train, Xs_train, y_train,
            Xc_val=Xc_val, Xs_val=Xs_val, y_val=y_val,
            model_args={
                "base_channels": a,
                "depth":         2, # try depth=2 # depth=3 gave 3.4e-01 after 199eps
                "kernel_size":   5,
                "batchnorm":     True,
                "dropout":       0.1,
                "scalar_embed":  b,
            },
            epochs=200, patience=15, batch_size=256,
            verbose=False
        )

        metrics = evaluate_conv_scalar(model, scaler, Xc_test, Xs_test, y_t_test)
        print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")
                


16 8
RMSE 3.4202e-05 | MAE 1.1577e-05 | R² 0.5383
16 16
RMSE 3.3991e-05 | MAE 1.1402e-05 | R² 0.5395
16 32
RMSE 3.4152e-05 | MAE 1.1116e-05 | R² 0.5440
16 64
RMSE 3.4357e-05 | MAE 1.1378e-05 | R² 0.5296
32 8


Your existing evaluate_conv_scalar and predict_conv_scalar work on this unchanged — same input shapes, same scaler keys, same forward signature. That's deliberate, so all your models go through one harness.